In [1]:
import sys
import os
from astropy.io import fits
import pandas as pd

# ====================================================================
# 1. ENLAZAR EL DIRECTORIO EXTERNO
# ====================================================================
ruta_directorio_clase = "MAIN_CODE/" 

if ruta_directorio_clase not in sys.path:
    sys.path.append(ruta_directorio_clase)

# ====================================================================
# 2. IMPORTAR LA CLASE
# ====================================================================
from Morphology_analysis import MorphologyAnalyzer


In [2]:

# ====================================================================
# 3. INSTANCIAR LA CLASE
# ====================================================================
print("Inicializando el analizador morfológico...")
analizador = MorphologyAnalyzer()

Inicializando el analizador morfológico...


In [3]:
# Sacar la informacion del CSV
ruta_info_fits = "/path/to/your/data/CFHT/buxs_wise_sources_analysis.fits"
with fits.open(ruta_info_fits) as hdul:
    data = hdul[1].data
    df_info = pd.DataFrame(data); print(df_info.head())


           designation  wise_src_index      followup  SRC_NUM  SRC_INDEX  \
0  J130552.64+175203.0             3.0        GTC15A        3          3   
1  J130955.66+321122.7             5.0        GTC15A        5          5   
2  J131048.44+322253.4             7.0   DESI,WHT14A        7          7   
3  J124325.16+112442.6             9.0            -1        9          9   
4  J140944.08+261220.0            32.0  WHT14A, DESI       32         32   

  2XMMi_SRCID_BUXS  SRC_INDEX_BUXS  redshift  zErr  class  ...  best_Ks_2mass  \
0               -1              -1     1.100  -1.0      3  ...       0.065682   
1               -1              -1     0.380  -1.0      2  ...       0.081363   
2               -1              -1     1.900  -1.0      4  ...       0.316975   
3               -1              -1    -1.000  -1.0     -1  ...            NaN   
4               -1              -1     1.262  -1.0      3  ...       0.174840   

   best_ukidss_y  best_ukidss_j  best_ukidss_h  best_uki

In [4]:
import numpy as np
from astropy.io import fits
import os

# ====================================================================
# 4. EJECUTAR EL MÉTODO DE SEGMENTACIÓN CON RECORTE FÍSICO (INLINE)
# ====================================================================
# Definimos los datos de prueba y rutas
mi_imagen_fits = "test/data_test/NGVS+0-1.G_cutout_3arcmin.fits"
weight_map_fits = "test/data_test/NGVS+0-1.G_weight_cutout_3arcmin.fits"
mi_directorio_salida = "test/data_test/"

# File data
WISE_index = 199
RA = df_info[df_info['SRC_INDEX'] == 199]['ra'].values[0]  # RA de la galaxia (en grados)
DEC = df_info[df_info['SRC_INDEX'] == 199]['dec'].values[0]  # DEC de la galaxia (en grados)
REDSHIFT = df_info[df_info['SRC_INDEX'] == 199]['redshift'].values[0]  # Redshift de la galaxia

# Nos aseguramos de que el directorio de salida exista
os.makedirs(mi_directorio_salida, exist_ok=True)

# --------------------------------------------------------------------
# Parámetros físicos del objeto y del instrumento
# --------------------------------------------------------------------
ra_agn = RA         # <-- Pon aquí la RA real de tu galaxia
dec_agn = DEC          # <-- Pon aquí la DEC real de tu galaxia
redshift_agn = REDSHIFT       # <-- Redshift (z) del objeto
radio_recorte_kpc = 80.0  # <-- Radio del recorte en kpc
pixel_scale_gtc = 0.254   # Escala de placa para GTC (arcsec/px)

try:
    # Abrimos el archivo FITS para extraer los datos y la cabecera (header)
    with fits.open(mi_imagen_fits) as hdul:
        if hdul[0].data is not None:
            image_data = hdul[0].data
            header = hdul[0].header
        else:
            image_data = hdul[1].data
            header = hdul[1].header
            
    # ====================================================================
    # PROCESO DE RECORTE DIRECTO USANDO TUS FUNCIONES EXISTENTES
    # ====================================================================
    # 1. Buscamos el centro geométrico (x, y) de la galaxia en píxeles
    px, py = analizador.radec_to_pixel(header, ra_agn, dec_agn)
    center_x, center_y = int(np.round(px)), int(np.round(py))
    
    # 2. Traducimos los kpc a píxeles usando tu función physical_to_pixels
    # NOTA: Tu función espera parsecs (pc), por eso multiplicamos los kpc por 1000
    pixel_size_val = analizador.physical_to_pixels(
        size_pc=radio_recorte_kpc * 1000, 
        redshift=redshift_agn, 
        pixel_scale_arcsec=pixel_scale_gtc
    )
    crop_radius = round(int(np.ceil(pixel_size_val))) - 1
    
    # 3. Calculamos los límites de la caja asegurando no salirnos de la imagen
    y_min = max(0, center_y - crop_radius)
    y_max = min(image_data.shape[0], center_y + crop_radius)
    x_min = max(0, center_x - crop_radius)
    x_max = min(image_data.shape[1], center_x + crop_radius)
    
    # 4. Cortamos la matriz para obtener la imagen pequeña centrada
    imagen_recortada = image_data[y_min:y_max, x_min:x_max]
    # ====================================================================

    # Si el recorte se ejecutó correctamente, procedemos a la segmentación
    if imagen_recortada is not None and imagen_recortada.size > 0:
        
        # Llamamos a tu función de segmentación pasando la IMAGEN RECORTADA
        resultados_segmentacion = analizador.SegmentationMap(
            image=imagen_recortada,  
            hot_snr=30.0,    
            cold_snr=2.0
        )

        # Si todo ha ido bien y no ha devuelto 'None'
        if resultados_segmentacion is not None:
            print("\n--- ¡Análisis completado! ---")
            
            imagen_limpia = resultados_segmentacion['imagen_limpia']
            mapa_de_ruido = resultados_segmentacion['ruido_mapa']
            mascara_total = resultados_segmentacion['segmask_total']
            mapa_final = resultados_segmentacion['segmap_final']
            
            print(f"El mapa de segmentación detectó {len(mapa_final.labels)} objetos distintos.")
            
            # Graficamos los resultados en el panel 2x2
            analizador.plot_segmentation_results(
                seg_results=resultados_segmentacion, 
                obj_id=f"GTC69-25B (z={redshift_agn}, R={radio_recorte_kpc}kpc)" 
            )

except FileNotFoundError:
    print(f"Error: No se ha encontrado el archivo en la ruta {mi_imagen_fits}")
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")

 [SegmentationMap] Preparando imagen y máscaras iniciales...
 [SegmentationMap] Estimando fondo del cielo iterativamente...
 [SegmentationMap] Generando mapa de segmentación (Hot-Cold)...


the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]


 [SegmentationMap] ¡Segmentación completada con éxito!

--- ¡Análisis completado! ---
El mapa de segmentación detectó 1 objetos distintos.
Ha ocurrido un error inesperado: MorphologyAnalyzer.plot_segmentation_results() missing 1 required positional argument: 'plots_dir'


In [5]:
# Example Morphology Parameter Calculation
results = analizador.calculate_morphology_parameters(
    obj_id=f"WISE{df_info[df_info['SRC_INDEX'] == 199]['SRC_INDEX'].values[0]}",         # ID of the object
    file_path=mi_imagen_fits,              # We compute the morphology on the unsharp masked image
    ra=df_info[df_info['SRC_INDEX'] == 199]['ra'].values[0],             # RA of the object
    dec=df_info[df_info['SRC_INDEX'] == 199]['dec'].values[0],           # DEC of the object
    redshift=df_info[df_info['SRC_INDEX'] == 199]['redshift'].values[0], # redshift of the object
    size_kpc=60.0,                         # size of the image to analyze in kpc
    gain=None,
    weightmap=weight_map_fits,  # <-- Aquí pasamos la ruta del mapa de pesos
    plots_dir=mi_directorio_salida   # base output directory
)


 =========== PROCESSING MORPHOLOGY FOR WISE199 =========== 

 [Crop] Calculando recorte para 60.0 kpc a z=0.946...
 [Crop] Centro del AGN (x,y): (484, 484)
 [Crop] Radio de recorte: 29 píxeles.
  ----> Crop radius: 29 pixels (from 60.0 kpc at z=0.946)


the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]


 [SegmentationMap] Preparando imagen y máscaras iniciales...
 [SegmentationMap] Estimando fondo del cielo iterativamente...
 [SegmentationMap] Generando mapa de segmentación (Hot-Cold)...
 [SegmentationMap] ¡Segmentación completada con éxito!



 [✓] Weightmap loaded successfully. 
 Running statmorph with weightmap ...

  Isophote asymmetry values for level:
    Back. std Level = 0.2723 |  A_x = 0.4243
    Back. std Level = 0.5447 |  A_x = 0.3220
    Back. std Level = 1.6341 |  A_x = 0.4337
    Back. std Level = 2.7234 |  A_x = 0.5724
    Back. std Level = 4.9022 |  A_x = 0.2658

  ----> Statmorph running time: 0.586717 s. 

  Saved Shape Asymmetry plot to test/data_test/Analyzed/A_s_Figs/WISE199_As.png 


  [✓]  Morphology parameters calculated successfully
 
=========== WISE199 MORPHOLOGICAL RESULTS ===========
  Redshift: 0.9460
  Concentration (C): 2.6327
  Asymmetry (A): 0.1569
  Smoothness (S): -0.0483
  Shape_Asymmetry (A_s): 0.3158
  Gini: 0.5192
  M20: -1.5251
